In [ ]:
import numpy as np

# Data Loading and Preparation

In [ ]:
from sklearn.datasets import fetch_openml
iris = fetch_openml(name='iris', version=1)

In [ ]:
print(iris.data.shape)
print(iris.target.shape)
print(np.unique(iris.target))
print(iris.DESCR)

(150, 4)
(150,)
['Iris-setosa' 'Iris-versicolor' 'Iris-virginica']
**Author**: R.A. Fisher  
**Source**: [UCI](https://archive.ics.uci.edu/ml/datasets/Iris) - 1936 - Donated by Michael Marshall  
**Please cite**:   

**Iris Plants Database**  
This is perhaps the best known database to be found in the pattern recognition literature.  Fisher's paper is a classic in the field and is referenced frequently to this day.  (See Duda & Hart, for example.)  The data set contains 3 classes of 50 instances each, where each class refers to a type of iris plant.  One class is     linearly separable from the other 2; the latter are NOT linearly separable from each other.

Predicted attribute: class of iris plant.  
This is an exceedingly simple domain.  
 
### Attribute Information:
    1. sepal length in cm
    2. sepal width in cm
    3. petal length in cm
    4. petal width in cm
    5. class: 
       -- Iris Setosa
       -- Iris Versicolour
       -- Iris Virginica

Downloaded from openml.org.


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target,
                                       test_size=0.20,
                                       random_state=32,
                                       shuffle=True)

In [ ]:
print(np.shape(X_train))
print(np.shape(X_test))
print(np.shape(y_train))
print(np.shape(y_test))

(120, 4)
(30, 4)
(120,)
(30,)


# decision tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier()
clf.fit(X_train, y_train)

DecisionTreeClassifier()

In [ ]:
# Calculate the accuracy of the model
predicted = clf.predict(X_test)
print(predicted)

['Iris-versicolor' 'Iris-setosa' 'Iris-setosa' 'Iris-versicolor'
 'Iris-virginica' 'Iris-virginica' 'Iris-setosa' 'Iris-setosa'
 'Iris-versicolor' 'Iris-setosa' 'Iris-versicolor' 'Iris-virginica'
 'Iris-versicolor' 'Iris-versicolor' 'Iris-virginica' 'Iris-virginica'
 'Iris-versicolor' 'Iris-virginica' 'Iris-versicolor' 'Iris-setosa'
 'Iris-setosa' 'Iris-virginica' 'Iris-virginica' 'Iris-setosa'
 'Iris-setosa' 'Iris-versicolor' 'Iris-setosa' 'Iris-virginica'
 'Iris-setosa' 'Iris-setosa']


## Evaluating Accuracy, Precision, Recall and F-measure of a model

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
cm = confusion_matrix(y_test, predicted)
cm

array([[12,  0,  0],
       [ 0,  9,  0],
       [ 0,  0,  9]])

In [ ]:
print(classification_report(y_test, predicted))

                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        12
Iris-versicolor       1.00      1.00      1.00         9
 Iris-virginica       1.00      1.00      1.00         9

       accuracy                           1.00        30
      macro avg       1.00      1.00      1.00        30
   weighted avg       1.00      1.00      1.00        30



## defining decision tree from scratch

In [ ]:
def entropy(y):
    hist = np.bincount(y) #storing frequency of each class where class is the index
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

In [ ]:
unique_classes, y_trainc = np.unique(y_train, return_inverse=True)
y_trainc

array([1, 0, 0, 2, 1, 0, 0, 2, 2, 1, 0, 2, 0, 2, 0, 2, 1, 1, 1, 0, 1, 1,
       0, 2, 1, 0, 0, 0, 0, 1, 2, 2, 1, 1, 2, 2, 0, 2, 0, 0, 2, 0, 2, 0,
       2, 0, 1, 0, 0, 1, 1, 2, 0, 1, 0, 1, 0, 2, 1, 1, 2, 1, 0, 1, 2, 2,
       1, 2, 0, 2, 2, 2, 0, 2, 1, 2, 1, 2, 1, 1, 1, 0, 2, 2, 1, 1, 2, 0,
       2, 1, 2, 2, 1, 1, 0, 1, 2, 2, 1, 0, 1, 2, 2, 1, 0, 0, 0, 1, 2, 2,
       0, 0, 0, 1, 1, 1, 2, 1, 2, 0])

In [ ]:
#splits the column values based on threshold
def split(X_column, thresh):
    left_idxs = np.argwhere(X_column <= thresh).flatten()
    right_idxs = np.argwhere(X_column > thresh).flatten()
    return left_idxs, right_idxs

In [ ]:
def information_gain(y, X_column, split_thresh):
    parent_entropy = entropy(y)

    #find the partitions
    left_idxs, right_idxs = split(X_column, split_thresh)

    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0

    n = len(y)
    n_left, n_right = len(left_idxs), len(right_idxs) #number of samples in each partition

    e_left, e_right = entropy(y[left_idxs]), entropy(y[right_idxs]) # entropy of each partition
    child_entropy = (n_left / n) * e_left + (n_right / n) * e_right

    ig = parent_entropy - child_entropy
    return ig

In [ ]:
# Find the best split
def best_split_values(X, y, feat_idxs):
    best_gain = -1
    split_idx, split_thresh = None, None
    for feat_idx in feat_idxs:
        X_column = X.iloc[:, feat_idx]

        thresholds = np.unique(X_column) #finds the unique values of the feature
        for threshold in thresholds:
            gain = information_gain(y, X_column, threshold)
            if gain > best_gain:
                best_gain = gain
                split_idx = feat_idx
                split_thresh = threshold
    return split_idx, split_thresh

In [ ]:
n_samples, n_features = X_train.shape
n_labels = len(np.unique(y_trainc))

feat_idxs = np.random.choice(n_features, n_features, replace=False)
print("feat_idxs"+str(feat_idxs))

best_feat, best_thresh = best_split_values(X_train, y_trainc, feat_idxs)
print(best_feat, best_thresh)

feat_idxs[0 3 1 2]
3 0.6


## completing the decision tree

In [ ]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

In [ ]:
def grow_tree(X, y, depth=0):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))

    # Stop if conditions are met and assign the leaf node corresponding class label
    if depth >= max_depth or n_labels == 1 or n_samples < min_samples_split:
        leaf_value = np.argmax(np.bincount(y))
        return Node(value=leaf_value)

    feat_idxs = np.random.choice(n_features, n_features, replace=False)

    # Find the best split
    best_feat, best_thresh = best_split_values(X, y, feat_idxs)
    #print("feat and thres")
    #print(best_feat, best_thresh)
    left_idxs, right_idxs = split(X.iloc[:, best_feat], best_thresh)

    #recursively grow the tree on both sides
    left = grow_tree(X.iloc[left_idxs, :], y[left_idxs], depth + 1)
    right = grow_tree(X.iloc[right_idxs, :], y[right_idxs], depth + 1)
    return Node(best_feat, best_thresh, left, right)

In [ ]:
min_samples_split=2
max_depth=10
root = grow_tree(X_train, y_trainc)

In [ ]:
def traverse_tree(x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return traverse_tree(x, node.left)
        return traverse_tree(x, node.right)

predicted = np.array([traverse_tree(x, root) for x in X_test.values])
print(predicted)

[1 0 0 2 2 2 0 0 2 0 1 2 1 1 2 2 1 2 1 0 0 2 2 0 0 1 0 2 0 0]


In [ ]:
predicted_c = unique_classes[predicted]

In [ ]:
predicted_c

array(['Iris-versicolor', 'Iris-setosa', 'Iris-setosa', 'Iris-virginica',
       'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-setosa',
       'Iris-virginica', 'Iris-setosa', 'Iris-versicolor',
       'Iris-virginica', 'Iris-versicolor', 'Iris-versicolor',
       'Iris-virginica', 'Iris-virginica', 'Iris-versicolor',
       'Iris-virginica', 'Iris-versicolor', 'Iris-setosa', 'Iris-setosa',
       'Iris-virginica', 'Iris-virginica', 'Iris-setosa', 'Iris-setosa',
       'Iris-versicolor', 'Iris-setosa', 'Iris-virginica', 'Iris-setosa',
       'Iris-setosa'], dtype=object)

In [ ]:
cm = confusion_matrix(y_test, predicted_c)
cm

array([[12,  0,  0],
       [ 0,  7,  2],
       [ 0,  0,  9]])

In [ ]:
print(classification_report(y_test, predicted_c))

                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        12
Iris-versicolor       1.00      0.78      0.88         9
 Iris-virginica       0.82      1.00      0.90         9

       accuracy                           0.93        30
      macro avg       0.94      0.93      0.92        30
   weighted avg       0.95      0.93      0.93        30



# Task1: Apply (user-defined) decision tree for mouse and breast-cancer dataset

In [ ]:
mice_data = fetch_openml(name='miceprotein', version=4)
print(mice_data.data.shape)
print(mice_data.target.shape)
print(np.unique(mice_data.target))
print(mice_data.DESCR)

(1080, 77)
(1080,)
['c-CS-m' 'c-CS-s' 'c-SC-m' 'c-SC-s' 't-CS-m' 't-CS-s' 't-SC-m' 't-SC-s']
**Author**: Clara Higuera, Katheleen J. Gardiner, Krzysztof J. Cios  
**Source**: [UCI](https://archive.ics.uci.edu/ml/datasets/Mice+Protein+Expression) - 2015   
**Please cite**: Higuera C, Gardiner KJ, Cios KJ (2015) Self-Organizing Feature Maps Identify Proteins Critical to Learning in a Mouse Model of Down Syndrome. PLoS ONE 10(6): e0129126.

Expression levels of 77 proteins measured in the cerebral cortex of 8 classes of control and Down syndrome mice exposed to context fear conditioning, a task used to assess associative learning.

The data set consists of the expression levels of 77 proteins/protein modifications that produced detectable signals in the nuclear fraction of cortex. There are 38 control mice and 34 trisomic mice (Down syndrome), for a total of 72 mice. In the experiments, 15 measurements were registered of each protein per sample/mouse. Therefore, for control mice, there ar

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Helper function: Decision tree node
class TreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

# Helper function: Convert DataFrame to numeric
def convert_df_to_numeric(df):

    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            # Convert non-numeric columns to categorical codes
            df[col] = pd.factorize(df[col].astype(str))[0]
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

# Function to calculate gini impurity
def gini_impurity(y):
    proportions = np.bincount(y) / len(y)
    return 1 - np.sum(proportions**2)

# Function to split data
def split_data(X, y, feature, threshold):
    left_mask = X[:, feature] <= threshold
    right_mask = ~left_mask
    return X[left_mask], X[right_mask], y[left_mask], y[right_mask]

# Function to find the best split
def best_split(X, y):
    best_gini = float('inf')
    best_feature, best_threshold = None, None
    for feature in range(X.shape[1]):
        thresholds = np.unique(X[:, feature])
        for threshold in thresholds:
            _, _, y_left, y_right = split_data(X, y, feature, threshold)
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            gini = (len(y_left) * gini_impurity(y_left) + len(y_right) * gini_impurity(y_right)) / len(y)
            if gini < best_gini:
                best_gini = gini
                best_feature, best_threshold = feature, threshold
    return best_feature, best_threshold

# Recursive function to grow the tree
def grow_tree(X, y, depth=0, max_depth=10, min_samples_split=2):
    if len(np.unique(y)) == 1 or len(y) < min_samples_split or depth == max_depth:
        return TreeNode(value=np.argmax(np.bincount(y)))
    feature, threshold = best_split(X, y)
    if feature is None:
        return TreeNode(value=np.argmax(np.bincount(y)))
    left_X, right_X, left_y, right_y = split_data(X, y, feature, threshold)
    left_child = grow_tree(left_X, left_y, depth + 1, max_depth, min_samples_split)
    right_child = grow_tree(right_X, right_y, depth + 1, max_depth, min_samples_split)
    return TreeNode(feature=feature, threshold=threshold, left=left_child, right=right_child)

# Function to traverse the tree for predictions
def traverse_tree(x, node):
    if node.value is not None:
        return node.value
    if x[node.feature] <= node.threshold:
        return traverse_tree(x, node.left)
    else:
        return traverse_tree(x, node.right)

# Mice Protein Dataset
mice_data = fetch_openml(name='MiceProtein', version=4)
X_train_mice, X_test_mice, y_train_mice, y_test_mice = train_test_split(
    mice_data.data, mice_data.target, test_size=0.20, random_state=32, shuffle=True
)

# Preprocess training and test data for Mice Protein dataset
df_train_mice = convert_df_to_numeric(pd.DataFrame(X_train_mice))
df_test_mice = convert_df_to_numeric(pd.DataFrame(X_test_mice))

# Factorize target labels for Mice Protein
unique_classes_mice, y_train_mice_c = np.unique(y_train_mice, return_inverse=True)

# Grow the decision tree for Mice Protein dataset
root_mice = grow_tree(df_train_mice.to_numpy(), y_train_mice_c)

# Make predictions on the Mice Protein test set
predicted_mice = np.array([traverse_tree(x, root_mice) for x in df_test_mice.to_numpy()])
predicted_mice_c = unique_classes_mice[predicted_mice]

# Evaluate Mice Protein model
cm_mice = confusion_matrix(y_test_mice, predicted_mice_c)
print("Confusion Matrix for Mice Protein Dataset:\n", cm_mice)
print(classification_report(y_test_mice, predicted_mice_c))

# Breast Cancer Dataset
breast_cancer_data = fetch_openml(name='breast-cancer', version=1)
X_train_bc, X_test_bc, y_train_bc, y_test_bc = train_test_split(
    breast_cancer_data.data, breast_cancer_data.target, test_size=0.20, random_state=32, shuffle=True
)

# Preprocess training and test data for Breast Cancer dataset
df_train_bc = convert_df_to_numeric(pd.DataFrame(X_train_bc))
df_test_bc = convert_df_to_numeric(pd.DataFrame(X_test_bc))

# Factorize target labels for Breast Cancer
unique_classes_bc, y_train_bc_c = np.unique(y_train_bc, return_inverse=True)

# Grow the decision tree for Breast Cancer dataset
root_bc = grow_tree(df_train_bc.to_numpy(), y_train_bc_c)

# Make predictions on the Breast Cancer test set
predicted_bc = np.array([traverse_tree(x, root_bc) for x in df_test_bc.to_numpy()])
predicted_bc_c = unique_classes_bc[predicted_bc]

# Evaluate Breast Cancer model
cm_bc = confusion_matrix(y_test_bc, predicted_bc_c)
print("Confusion Matrix for Breast Cancer Dataset:\n", cm_bc)
print(classification_report(y_test_bc, predicted_bc_c))


Confusion Matrix for Mice Protein Dataset:
 [[25  4  0  0  4  1  0  0]
 [ 2 16  0  0  2  2  0  0]
 [ 0  0 36  1  0  0  2  1]
 [ 0  0  0 22  0  0  0  0]
 [ 1  1  0  0 26  1  0  0]
 [ 0  1  0  0  1 14  0  0]
 [ 0  0  1  3  0  0 24  0]
 [ 0  0  1  0  0  0  1 23]]
              precision    recall  f1-score   support

      c-CS-m       0.89      0.74      0.81        34
      c-CS-s       0.73      0.73      0.73        22
      c-SC-m       0.95      0.90      0.92        40
      c-SC-s       0.85      1.00      0.92        22
      t-CS-m       0.79      0.90      0.84        29
      t-CS-s       0.78      0.88      0.82        16
      t-SC-m       0.89      0.86      0.87        28
      t-SC-s       0.96      0.92      0.94        25

    accuracy                           0.86       216
   macro avg       0.85      0.86      0.86       216
weighted avg       0.87      0.86      0.86       216

Confusion Matrix for Breast Cancer Dataset:
 [[26 16]
 [ 8  8]]
                      pr

In [ ]:
cancer_data = fetch_openml(name='breast-cancer', version=1)
print(cancer_data.data.shape)
print(cancer_data.target.shape)
print(np.unique(cancer_data.target))
print(cancer_data.DESCR)

(286, 9)
(286,)
['no-recurrence-events' 'recurrence-events']
**Author**:   
**Source**: Unknown -   
**Please cite**:   

Citation Request:
    This breast cancer domain was obtained from the University Medical Centre,
    Institute of Oncology, Ljubljana, Yugoslavia.  Thanks go to M. Zwitter and 
    M. Soklic for providing the data.  Please include this citation if you plan
    to use this database.
 
 1. Title: Breast cancer data (Michalski has used this)
 
 2. Sources: 
    -- Matjaz Zwitter & Milan Soklic (physicians)
       Institute of Oncology 
       University Medical Center
       Ljubljana, Yugoslavia
    -- Donors: Ming Tan and Jeff Schlimmer (Jeffrey.Schlimmer@a.gp.cs.cmu.edu)
    -- Date: 11 July 1988
 
 3. Past Usage: (Several: here are some)
      -- Michalski,R.S., Mozetic,I., Hong,J., & Lavrac,N. (1986). The 
         Multi-Purpose Incremental Learning System AQ15 and its Testing 
         Application to Three Medical Domains.  In Proceedings of the 
         Fifth N

# Task2: Implement (user-defined) gain ratio based decision tree and evaluate with Iris dataset

In [ ]:

import numpy as np
import pandas as pd
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# Data Loading and Preparation
iris = fetch_openml(name='iris', version=1)
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=32, shuffle=True)

def entropy(y):
    hist = np.bincount(y)
    ps = hist / len(y)
    return -np.sum([p * np.log2(p) for p in ps if p > 0])

def split(X_column, thresh):
    left_idxs = np.argwhere(X_column <= thresh).flatten()
    right_idxs = np.argwhere(X_column > thresh).flatten()
    return left_idxs, right_idxs

def information_gain(y, X_column, split_thresh):
    parent_entropy = entropy(y)
    left_idxs, right_idxs = split(X_column, split_thresh)

    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0

    n = len(y)
    n_left, n_right = len(left_idxs), len(right_idxs)
    e_left, e_right = entropy(y[left_idxs]), entropy(y[right_idxs])
    child_entropy = (n_left / n) * e_left + (n_right / n) * e_right
    ig = parent_entropy - child_entropy
    return ig

def split_info(y, X_column, split_thresh):
    left_idxs, right_idxs = split(X_column,split_thresh)
    if len(left_idxs) == 0 or len(right_idxs) == 0:
        return 0
    n = len(y)
    n_left, n_right = len(left_idxs), len(right_idxs)
    return - (n_left/n)*np.log2(n_left/n) - (n_right/n)*np.log2(n_right/n)


def gain_ratio(y, X_column, split_thresh):
    info_gain_val = information_gain(y, X_column, split_thresh)
    split_info_val = split_info(y, X_column, split_thresh)
    if split_info_val == 0:
      return 0
    return info_gain_val / split_info_val


def best_split_values(X, y, feat_idxs):
    best_gain = -1
    split_idx, split_thresh = None, None
    for feat_idx in feat_idxs:
        X_column = X.iloc[:, feat_idx]
        thresholds = np.unique(X_column)
        for threshold in thresholds:
            gain = gain_ratio(y, X_column, threshold)
            if gain > best_gain:
                best_gain = gain
                split_idx = feat_idx
                split_thresh = threshold
    return split_idx, split_thresh

unique_classes, y_trainc = np.unique(y_train, return_inverse=True)
n_samples, n_features = X_train.shape
n_labels = len(np.unique(y_trainc))

min_samples_split = 2
max_depth = 10

feat_idxs = np.random.choice(n_features, n_features, replace=False)
root = grow_tree(X_train, y_trainc)
predicted = np.array([traverse_tree(x, root) for x in X_test.values])
predicted_c = unique_classes[predicted]

cm = confusion_matrix(y_test, predicted_c)
print(cm)
print(classification_report(y_test, predicted_c))


[[12  0  0]
 [ 0  7  2]
 [ 0  1  8]]
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        12
Iris-versicolor       0.88      0.78      0.82         9
 Iris-virginica       0.80      0.89      0.84         9

       accuracy                           0.90        30
      macro avg       0.89      0.89      0.89        30
   weighted avg       0.90      0.90      0.90        30

